In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [13]:
df = pd.read_csv('../Data/train-2026.csv')
print(df.shape)
df.head(5)

/var/folders/22/3z4yr_bd5ds4d1g7ph9nprs80000gn/T/ipykernel_69996/3753667672.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../Data/train-2026.csv')


(1023552, 44)


,Unnamed: 0,id,stmt_date,HQ_city,legal_struct,ateco_sector,def_date,fs_year,asst_intang_fixed,asst_tang_fixed,...,inc_extraord,taxes,profit,days_rec,ebitda,roa,roe,wc_net,margin_fin,cf_operations
0,17,520288,2011-12-31,28.0,SRL,14.0,NaN,2011,67537.0,1137566.0,...,-309.0,-80959.0,-81288.0,NaN,6318.0,-3.81,-28.03,496258.0,-917029.0,-849.0
1,18,520288,2008-12-31,28.0,SRL,14.0,NaN,2008,256438.0,1181416.0,...,-678.0,-94622.0,-107382.0,NaN,46088.0,-2.76,NaN,97952.0,NaN,-3881.0
2,19,520288,2009-12-31,28.0,SRL,14.0,NaN,2009,194046.0,1152014.0,...,4224.0,-74235.0,-77819.0,NaN,67611.0,-2.17,NaN,-210671.0,NaN,32618.0
3,21,520288,2012-12-31,28.0,SRL,14.0,NaN,2012,15195.0,1116938.0,...,3634.0,-250786.0,-250786.0,NaN,-161478.0,-12.99,NaN,367892.0,-1094962.0,-168907.0
4,22,520288,2007-12-31,28.0,SRL,14.0,NaN,2007,126603.0,1127807.0,...,820.0,92192.0,36733.0,NaN,153060.0,6.20,52.43,-317007.0,-1184970.0,80039.0


In [14]:
identifier_and_date_cols = ['id', 'stmt_date', 'def_date', 'fs_year']
firm_characteristics_cols = ['HQ_city', 'legal_struct', 'ateco_sector']
asset_cols = ['asst_intang_fixed', 'asst_tang_fixed', 'asst_fixed_fin', 'asst_current', 'AR', 'cash_and_equiv', 'asst_tot', 'goodwill']
equity_capital_cols = ['eqty_tot', 'eqty_corp_family_tot', 'margin_fin']
debt_cols = ['liab_lt', 'liab_lt_emp', 'debt_bank_st', 'debt_bank_lt', 'debt_fin_st', 'debt_fin_lt', 'AP_st', 'AP_lt', 'debt_st', 'debt_lt']
revenue_and_expenses_cols = ['rev_operating', 'COGS', 'inc_financing', 'exp_financing', 'inc_extraord', 'taxes']
profitability_cols = ['prof_operations', 'prof_financing', 'profit', 'ebitda', 'roa', 'roe']
liquidity_and_cashflows_cols = ['wc_net', 'cf_operations']
ops_efficiency_cols = ['days_rec']

In [15]:
column_mapping = {
    'identifier_and_date_cols' : identifier_and_date_cols,
    'firm_characteristics_cols' : firm_characteristics_cols,
    'asset_cols' : asset_cols,
    'equity_capital_cols' : equity_capital_cols,
    'debt_cols' : debt_cols,
    'revenue_and_expenses_cols' : revenue_and_expenses_cols,
    'profitability_cols' : profitability_cols,
    'liquidity_and_cashflows_cols' : liquidity_and_cashflows_cols,
    'ops_efficiency_cols' : ops_efficiency_cols
}

In [16]:
import pickle

pickle.dump(column_mapping, open('../Files/column_mapping.pkl', 'wb'))

In [17]:
data_statistics = {}
for _k, _v in column_mapping.items():
    data_statistics[_k] = {}
    data_statistics[_k]['description'] = df[_v].describe()
    null_counts = pd.DataFrame(df[_v].isna().sum()).reset_index()
    null_counts.columns = ['feature', 'null_count']
    data_statistics[_k]['null_information'] = null_counts

pickle.dump(data_statistics, open('../Files/data_statistics.pkl', 'wb'))

In [18]:
t_all = pd.DataFrame(columns=['feature','null_count'])
for _k in data_statistics.keys():
    t = data_statistics[_k]['null_information'].query('null_count > 0')
    t_all = pd.concat([t_all, t])

In [19]:
from utils import create_target_variable

df['default'] = df.apply(lambda row: create_target_variable(row['stmt_date'], row['def_date']), axis=1)
df['default'].isna().sum(), df.shape[0]
df = df.loc[~df['default'].isna()]

df.groupby(['fs_year']).agg({'id':'count','default':['sum', 'mean']})

id default          
          count     sum      mean
fs_year                          
2007     144128    1475  0.010234
2008     161136    2308  0.014323
2009     170125    2867  0.016852
2010     174229    2967  0.017029
2011     183730    2134  0.011615
2012     184989       0       0.0

In [20]:
df.to_csv('../Data/train-2026_with_target_variable.csv')

In [21]:
df['data_type'] = df['fs_year'].apply(lambda x: 'train' if x <= 2009 else ('oot1' if x <= 2010 else 'oot2'))
df = df.loc[df['fs_year'] != 2012]
df.groupby(['fs_year','data_type']).agg({'data_type':'count','default':'mean'})

,,data_type,default
fs_year,data_type,,
2007,train,144128,0.010234
2008,train,161136,0.014323
2009,train,170125,0.016852
2010,oot1,174229,0.017029
2011,oot2,183730,0.011615


In [22]:
df.to_csv('../Data/temp_training.csv')